# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, and columns) are referenced by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Published: {getattr(md, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(md, 'keywords', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced by their `@id`. Here, we enumerate all record sets and their corresponding field IDs.

In [ ]:
# Get all record set IDs
record_sets = dataset.list_record_sets()
print('Record Sets:')
for rs in record_sets:
    print(f"  - {rs}")

# For each record set, get its field IDs
record_set_fields = {}
for rs in record_sets:
    fields = dataset.list_fields(rs)
    record_set_fields[rs] = fields
    print(f"\nRecord Set {rs} fields:")
    for field in fields:
        print(f"    - {field}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Select one or more record sets (as an example, use first one)
chosen_record_set = record_sets[0]

# Load all records into a DataFrame
records = list(dataset.records(record_set=chosen_record_set))
df = pd.DataFrame(records)

print(f"Fields for Record Set {chosen_record_set}:")
print(df.columns.tolist())

print("\nFirst 5 records:")
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and grouping using field `@id`s. **Replace the `example_numeric_field_id` and `example_group_field` with appropriate field IDs as printed above.**

In [ ]:
# --- Example EDA with placeholder field IDs ---
# Please replace these with a real numeric field and (optionally) a grouping field from the above

# Example: Let's pick the first numeric-looking field for demonstration
candidate_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi']
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # If all columns are object, try coercing to numeric and checking for success
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col])
            numeric_field = col
            break
        except:
            continue
    else:
        numeric_field = df.columns[0]  # fallback: first column

# Filter for values > threshold
threshold = 10
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field
possible_groups = [col for col in df.columns if df[col].nunique() < min(10, len(df)//5) and col != numeric_field]
if possible_groups:
    group_field = possible_groups[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"{numeric_field}_mean")
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization

Here we visualize the distribution of a numeric field. Please adjust the field name as needed.

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram of the chosen numeric field
if numeric_field:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# If grouped data available, plot a bar graph
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 4))
    plt.bar(grouped_df[group_field].astype(str), grouped_df[f"{numeric_field}_mean"])
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Average {numeric_field}")
    plt.show()

## 6. Conclusion

- Successfully loaded the dataset schema and records using `mlcroissant`.
- Inspected record sets and field IDs (`@id`) per FAIR best practices.
- Extracted a main record set, performed simple filtering and normalization, and visualized a numeric field.

For more advanced analysis, use field IDs (`@id`) as referenced in the data overview, and refer to the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) documentation for scientific use.
